# Homework 10 — Spark Structured Streaming

Name: Alex Devoid  
Course: ST 554

This notebook is focued on structured streaming. I build a `rate` stream, add the square root of the `value` column and `value mod 4`, and write the result to an in-memory table. I then fit a Spark pipeline on the bike data, start a CSV stream on `HW10/bike_stream_input`, and use the fitted pipeline to transform bike CSV files as I add them to that folder.


In [ ]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
from IPython.display import display

# local pyspark setup
JAVA_HOME = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'
os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = f"{JAVA_HOME}/bin:{os.environ['PATH']}"
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


# Spark SQL, the streaming helper functions, and MLlib imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, pmod, sqrt
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler


pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:.4f}')


DIR = Path('/Users/alexdevoid/Documents/Stats/ST554-HW/HW10')
BIKE_FIT_DATA_PATH = DIR / 'bikeDetails_for_fit.csv'
WATCH_DIR = DIR / 'bike_stream_input'

# start Spark session
spark = SparkSession.builder.master('local[*]').appName('hw10_structured_streaming').getOrCreate()

# suppress routine Spark warning logs
spark.sparkContext.setLogLevel('ERROR')


## 1. Structured Streaming with `rate`

I use Spark's `rate` source to create a stream with a timestamp and integer `value`. I add square-root and mod transformations so the memory table stores the original and derived columns.

In [32]:
RATE_QUERY_NAME = 'hw10_rate_table'
# read the rate source
rate_stream = spark.readStream.format('rate').option('rowsPerSecond', 1).load()

# add the square-root and mod-4 columns 
rate_transformed = (
    rate_stream
    .withColumn('sqrt_value', sqrt(col('value')))
    .withColumn('value_mod_4', pmod(col('value'), lit(4)))
)

# write the transformed rate rows to an in-memory table.
rate_query = (
    rate_transformed
    .writeStream
    .format('memory')
    .queryName(RATE_QUERY_NAME)
    .outputMode('append')
    .start()
)

# let the query run for 30 seconds before stopping it
time.sleep(30)
rate_query.stop()

# read the full memory table after stopping the query
rate_output = spark.sql(f'SELECT * FROM {RATE_QUERY_NAME} ORDER BY value')

# count the stored rows, then print 
rate_output_count = rate_output.count()
print(f'Rows stored in {RATE_QUERY_NAME}: {rate_output_count}')
rate_output.show(rate_output_count, truncate=False)


Rows stored in hw10_rate_table: 30
+-----------------------+-----+------------------+-----------+
|timestamp              |value|sqrt_value        |value_mod_4|
+-----------------------+-----+------------------+-----------+
|2026-04-22 00:03:58.868|0    |0.0               |0          |
|2026-04-22 00:03:59.868|1    |1.0               |1          |
|2026-04-22 00:04:00.868|2    |1.4142135623730951|2          |
|2026-04-22 00:04:01.868|3    |1.7320508075688772|3          |
|2026-04-22 00:04:02.868|4    |2.0               |0          |
|2026-04-22 00:04:03.868|5    |2.23606797749979  |1          |
|2026-04-22 00:04:04.868|6    |2.449489742783178 |2          |
|2026-04-22 00:04:05.868|7    |2.6457513110645907|3          |
|2026-04-22 00:04:06.868|8    |2.8284271247461903|0          |
|2026-04-22 00:04:07.868|9    |3.0               |1          |
|2026-04-22 00:04:08.868|10   |3.1622776601683795|2          |
|2026-04-22 00:04:09.868|11   |3.3166247903554   |3          |
|2026-04-22 00:04:10

After 30 seconds, the memory table contains 30 streamed rows. The displayed table confirms that `sqrt_value` and `value_mod_4` are being computed directly from the original `value` column for each streamed row.

## 2. Fit the Bike Pipeline


In [33]:
# read the CSV data as a Spark SQL DataFrame
bike_fit = spark.read.option('header', True).option('inferSchema', True).csv(str(BIKE_FIT_DATA_PATH))

# SQL transformation to create the label and predictor columns.
bike_sql = SQLTransformer(statement="""
SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
FROM __THIS__
""")

# putting year, log_km_driven, and one_owner into the features vector
bike_assembler = VectorAssembler(
    inputCols=['year', 'log_km_driven', 'one_owner'],
    outputCol='features',
)

# wrap SQL transformation and assembler into a pipeline
bike_pipeline = Pipeline(stages=[bike_sql, bike_assembler])

# fit the pipeline on the bike data
bike_pipeline_model = bike_pipeline.fit(bike_fit)

# apply the fitted pipeline 
bike_fit_transformed = bike_pipeline_model.transform(bike_fit)

# size of the fit 
bike_fit_summary = pd.DataFrame(
    {
        'fit_rows': [bike_fit.count()],
        'fit_columns': [len(bike_fit.columns)],
    }
)
display(bike_fit_summary)

# show the transformed label, predictors, and assembled feature vector
bike_fit_transformed.select('label', 'year', 'log_km_driven', 'one_owner', 'features').show(5, truncate=False)


,fit_rows,fit_columns
0,758,7


+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|12.072541252905651|2019|5.857933154483459 |1        |[2019.0,5.857933154483459,1.0] |
|10.714417768752456|2017|8.639410824140487 |1        |[2017.0,8.639410824140487,1.0] |
|11.918390573078392|2018|9.392661928770137 |1        |[2018.0,9.392661928770137,1.0] |
|11.082142548877775|2015|10.043249494911286|1        |[2015.0,10.043249494911286,1.0]|
|9.903487552536127 |2011|9.95227771670556  |0        |[2011.0,9.95227771670556,0.0]  |
+------------------+----+------------------+---------+-------------------------------+
only showing top 5 rows


In [34]:
# use the fitted static schema for the incoming CSV stream
bike_stream = spark.readStream.schema(bike_fit.schema).option('header', True).csv(str(WATCH_DIR))

# apply the fitted pipeline to each incoming CSV batch.
bike_stream_transformed = bike_pipeline_model.transform(bike_stream)

# write the transformed stream to the console in append mode
bike_stream_query = (
    bike_stream_transformed
    .writeStream
    .format('console')
    .outputMode('append')
    .start()
)

print(f'Stream started')


Stream started


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [35]:
# stop the stream after moving the files and processing them
bike_stream_query.stop()

# stop Spark.
spark.stop()

The output shows five batches, and each batch includes the `label`, `log_km_driven`, `one_owner`, and the `features` columns, so the streamed files are being processed with the same SQL transformation and feature processing steps that I fit on `bikeDetails_for_fit.csv`.